# Air Pollution Forecasting

## Project Overview

Forecasts **daily Air Quality Index (AQI)** over a 14-day horizon. Synthetic data spans 2 years with weekly traffic patterns, seasonal inversions, and weather-driven pollution events.

**Models**: Naive, LazyPredict, FLAML, StatsForecast. **Horizon**: 14 days.

## Learning Objectives

1. Understand time-series patterns (trend, seasonality, noise)
2. Build naive and seasonal-naive baselines
3. Engineer lag and rolling features for tabular ML
4. Use LazyPredict for quick regression benchmarking
5. Apply FLAML AutoML (excluding XGBoost due to compatibility)
6. Use StatsForecast (AutoARIMA, AutoETS, SeasonalNaive)
7. Compare all approaches with MAE / RMSE / MAPE

## Problem Statement

Given historical daily AQI values, predict the next 14 days. Environmental agencies need pollution forecasts for public health advisories, traffic restrictions, and emission control measures.

## Why This Project Matters

Air pollution causes 7 million premature deaths annually (WHO). Accurate AQI forecasts enable timely health warnings, help vulnerable populations plan outdoor activities, and allow cities to implement proactive emission controls before pollution episodes worsen.

## Dataset Overview

Synthetic dataset:
- 730 days (2 years) of daily AQI values
- Weekly pattern (higher on weekdays due to traffic/industry)
- Seasonal variation (winter inversions trap pollution)
- Occasional pollution episodes (stagnant weather)
- Slight improvement trend (emission regulations)

| Column | Description |
|--------|-------------|
| `date` | Date |
| `aqi` | Daily Air Quality Index (0-500 scale) |

## Dataset Source and License Notes

Synthetically generated for educational purposes. No external download required.

## Environment Setup

In [1]:
import subprocess, importlib
for pkg in ['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'statsforecast', 'flaml', 'lazypredict']:
    try:
        importlib.import_module(pkg.replace('-', '_').split('[')[0])
    except ImportError:
        subprocess.check_call(['pip', 'install', '-q', pkg])
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import mean_absolute_error, mean_squared_error
print("Imports complete.")

Imports complete.


## Configuration / Constants

In [3]:
SEED = 42
N_POINTS = 730
HORIZON = 14
VAL_SIZE = 14
TEST_SIZE = 14
SEASON_LENGTH = 7
FREQ = 'D'
TARGET = 'aqi'
np.random.seed(SEED)
print(f"Config: {N_POINTS} points, horizon={HORIZON}, season={SEASON_LENGTH}")

Config: 730 points, horizon=14, season=7


## Dataset Generation

In [4]:
dates = pd.date_range(start='2022-01-01', periods=N_POINTS, freq='D')
t = np.arange(N_POINTS)

base = 75 - 0.01 * t  # slight improvement trend
weekly = np.zeros(N_POINTS)
for i in range(N_POINTS):
    dow = dates[i].dayofweek
    if dow <= 4: weekly[i] = 8  # weekday traffic/industry
    elif dow == 5: weekly[i] = -5
    else: weekly[i] = -10  # Sunday lowest

# Winter inversions trap pollution
seasonal = 20 * np.cos(2 * np.pi * (t - 15) / 365)

# Pollution episodes (stagnant weather)
episodes = np.zeros(N_POINTS)
for s in range(0, N_POINTS, 90):
    ep_start = min(s + np.random.randint(20, 60), N_POINTS - 4)
    for j in range(min(3, N_POINTS - ep_start)):
        episodes[ep_start + j] = 30 + np.random.uniform(0, 40)

noise = np.random.normal(0, 8, N_POINTS)
aqi = base + weekly + seasonal + episodes + noise
aqi = np.maximum(aqi, 10).round(0).astype(int)

df = pd.DataFrame({'date': dates, 'aqi': aqi})
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (730, 2)


,date,aqi
0,2022-01-01,85
1,2022-01-02,82
2,2022-01-03,98
3,2022-01-04,117
4,2022-01-05,102


## Data Validation Checks

In [5]:
assert df.isnull().sum().sum() == 0, "Missing values"
assert (df[TARGET] > 0).all(), "Non-positive values"
assert df['date'].is_monotonic_increasing, "Not sorted"
assert len(df) == N_POINTS, "Row count mismatch"
print("All validation checks passed.")

All validation checks passed.


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].plot(df['date'], df[TARGET], linewidth=0.6)
axes[0, 0].set_title('aqi Over Time')
axes[0, 1].hist(df[TARGET], bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Distribution')
from pandas.plotting import autocorrelation_plot
autocorrelation_plot(df[TARGET], ax=axes[1, 0])
axes[1, 0].set_xlim(0, min(60, N_POINTS // 2))
axes[1, 0].set_title('Autocorrelation')
rw = min(SEASON_LENGTH * 2, N_POINTS // 4)
axes[1, 1].plot(df['date'], df[TARGET].rolling(rw).mean())
axes[1, 1].set_title(f'Rolling {rw}-period Mean')
plt.tight_layout()
plt.savefig('eda.png', dpi=100, bbox_inches='tight')
plt.show()
print("EDA complete.")

EDA complete.


## Target Analysis

In [7]:
print("aqi Statistics:")
print(df[TARGET].describe())
print(f"\nCV: {df[TARGET].std() / df[TARGET].mean():.3f}")
print(f"Skewness: {df[TARGET].skew():.3f}")

aqi Statistics:
count    730.000000
mean      76.621918
std       20.422212
min       23.000000
25%       62.000000
50%       76.000000
75%       90.000000
max      184.000000
Name: aqi, dtype: float64

CV: 0.267
Skewness: 0.677


## Train / Validation / Test Split Strategy

Time-aware split (no shuffling):
- **Train**: all except last 28
- **Validation**: 14 points
- **Test**: last 14 points

In [8]:
train = df.iloc[:-(VAL_SIZE + TEST_SIZE)].copy()
val = df.iloc[-(VAL_SIZE + TEST_SIZE):-TEST_SIZE].copy()
test = df.iloc[-TEST_SIZE:].copy()
print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

Train: 702, Val: 14, Test: 14


## Preprocessing Strategy

Minimal preprocessing — tree models handle raw features. Features created next.

In [9]:
df_full = df.copy().sort_values('date').reset_index(drop=True)
print('Data ready.')

Data ready.


## Feature Engineering

In [10]:
def create_features(data):
    d = data.copy()
    d['dow'] = d['date'].dt.dayofweek
    d['month'] = d['date'].dt.month
    d['day'] = d['date'].dt.day
    d['week_of_year'] = d['date'].dt.isocalendar().week.astype(int)
    for lag in [1, 7, 14, 28]:
        d[f'lag_{lag}'] = d[TARGET].shift(lag)
    for w in [7, 14, 28]:
        d[f'rmean_{w}'] = d[TARGET].shift(1).rolling(w).mean()
        d[f'rstd_{w}'] = d[TARGET].shift(1).rolling(w).std()
    return d

df_feat = create_features(df_full).dropna().reset_index(drop=True)
feature_cols = [c for c in df_feat.columns if c not in ['date', TARGET]]
print(f"Features: {len(feature_cols)} columns, {len(df_feat)} rows")

Features: 14 columns, 702 rows


## Baseline Model — Naive Forecasts

In [11]:
def calc_metrics(y_true, y_pred, name=""):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1))) * 100
    print(f"{name:30s} | MAE: {mae:10.1f} | RMSE: {rmse:10.1f} | MAPE: {mape:5.2f}%")
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

results = []
naive_pred = np.full(TEST_SIZE, train[TARGET].iloc[-1])
results.append(calc_metrics(test[TARGET].values, naive_pred, "Naive (Last Value)"))

src = df_full[TARGET].values
ti = len(df_full) - TEST_SIZE
sn_pred = src[ti - SEASON_LENGTH:ti - SEASON_LENGTH + TEST_SIZE]
results.append(calc_metrics(test[TARGET].values, sn_pred, f"Seasonal Naive ({SEASON_LENGTH})"))

Naive (Last Value)             | MAE:       36.6 | RMSE:       49.4 | MAPE: 30.21%
Seasonal Naive (7)             | MAE:       23.2 | RMSE:       38.3 | MAPE: 17.67%


## LazyPredict Benchmark (Lag-Feature Tabular)

In [12]:
from lazypredict.Supervised import LazyRegressor

ct = len(df_feat) - TEST_SIZE
cv = ct - VAL_SIZE
X_tr = df_feat.iloc[:cv][feature_cols]
y_tr = df_feat.iloc[:cv][TARGET]
X_va = df_feat.iloc[cv:ct][feature_cols]
y_va = df_feat.iloc[cv:ct][TARGET]

reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None, predictions=True)
lp_models, lp_preds = reg.fit(X_tr, X_va, y_tr, y_va)
print("\nLazyPredict Top 10:")
print(lp_models.head(10).to_string())


LazyPredict Top 10:
                           Adjusted R-Squared  R-Squared       RMSE  Time Taken
Model                                                                          
KernelRidge                        647.247489 -48.711345  75.030369    0.060385
GaussianProcessRegressor            70.327403  -4.332877  24.574807    0.059653
DecisionTreeRegressor               40.079654  -2.006127  18.450707    0.011713
QuantileRegressor                   37.898540  -1.838349  17.928429    0.087772
Lars                                36.210105  -1.708470  17.513434    0.016354
DummyRegressor                      31.673162  -1.359474  16.346204    0.006692
MLPRegressor                        19.529720  -0.425363  12.704924    0.347653
LarsCV                              18.761668  -0.366282  12.438830    0.019471
OrthogonalMatchingPursuit           16.401016  -0.184694  11.582767    0.007853
AdaBoostRegressor                   13.180848   0.063012  10.300930    0.093448


## FLAML AutoML Run

In [13]:
from flaml import AutoML

automl = AutoML()
automl.fit(
    X_train=X_tr, y_train=y_tr,
    task='regression', time_budget=30, metric='mae',
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
print(f"FLAML best: {automl.best_estimator}")
flaml_val = automl.predict(X_va)
results.append(calc_metrics(y_va.values, flaml_val, f"FLAML ({automl.best_estimator})"))

X_te = df_feat.iloc[ct:][feature_cols]
y_te = df_feat.iloc[ct:][TARGET]
flaml_test = automl.predict(X_te)
results.append(calc_metrics(y_te.values, flaml_test, f"FLAML Test ({automl.best_estimator})"))

FLAML best: catboost
FLAML (catboost)               | MAE:        7.9 | RMSE:        8.5 | MAPE:  8.91%
FLAML Test (catboost)          | MAE:       19.5 | RMSE:       33.8 | MAPE: 14.36%


## StatsForecast — Statistical Forecasting

In [14]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, SeasonalNaive as SFSeasonalNaive

sf_df = df_full[['date', TARGET]].copy()
sf_df.columns = ['ds', 'y']
sf_df['unique_id'] = 'series1'
sf_df = sf_df[['unique_id', 'ds', 'y']]

sf_train = sf_df.iloc[:-TEST_SIZE]
sf = StatsForecast(
    models=[AutoARIMA(season_length=SEASON_LENGTH), AutoETS(season_length=SEASON_LENGTH),
            SFSeasonalNaive(season_length=SEASON_LENGTH)],
    freq=FREQ, n_jobs=1
)
sf.fit(sf_train)
sf_preds = sf.predict(h=TEST_SIZE).reset_index()

y_test_sf = test[TARGET].values
for col in ['AutoARIMA', 'AutoETS', 'SeasonalNaive']:
    if col in sf_preds.columns:
        results.append(calc_metrics(y_test_sf, sf_preds[col].values, f"SF {col}"))
print("StatsForecast complete.")

SF AutoARIMA                   | MAE:       22.6 | RMSE:       38.0 | MAPE: 17.15%
SF AutoETS                     | MAE:       20.3 | RMSE:       36.4 | MAPE: 14.49%
SF SeasonalNaive               | MAE:       24.6 | RMSE:       40.2 | MAPE: 18.81%
StatsForecast complete.


## Top 3 Model Selection

In [15]:
results_df = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)
print("All Results:")
print(results_df.to_string(index=False))
print("\nTop 3:")
print(results_df.head(3).to_string(index=False))

All Results:
                model       MAE      RMSE      MAPE
     FLAML (catboost)  7.870241  8.517635  8.905312
FLAML Test (catboost) 19.519237 33.783514 14.364327
           SF AutoETS 20.314257 36.440252 14.487526
         SF AutoARIMA 22.644999 38.039144 17.148409
   Seasonal Naive (7) 23.214286 38.266920 17.673954
     SF SeasonalNaive 24.571428 40.222595 18.808751
   Naive (Last Value) 36.571429 49.449830 30.208059

Top 3:
                model       MAE      RMSE      MAPE
     FLAML (catboost)  7.870241  8.517635  8.905312
FLAML Test (catboost) 19.519237 33.783514 14.364327
           SF AutoETS 20.314257 36.440252 14.487526


## Final Training and Evaluation of Top 3

In [16]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test['date'].values, test[TARGET].values, 'ko-', label='Actual', ms=4)
ax.plot(test['date'].values, flaml_test, 's--', label=f'FLAML ({automl.best_estimator})', ms=4)
for col in ['AutoARIMA', 'AutoETS']:
    if col in sf_preds.columns:
        ax.plot(test['date'].values[:len(sf_preds)], sf_preds[col].values, 'o--', label=f'SF {col}', ms=4)
ax.set_title('Forecast vs Actual — Test Set')
ax.legend()
plt.tight_layout()
plt.savefig('forecast_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## Error Analysis

In [17]:
errors = test[TARGET].values - flaml_test
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(len(errors)), errors, alpha=0.7)
axes[0].set_title('Residuals (Actual - FLAML)')
axes[0].axhline(y=0, color='r', linestyle='--')
axes[1].hist(errors, bins=10, edgecolor='black', alpha=0.7)
axes[1].set_title('Residual Distribution')
plt.tight_layout()
plt.savefig('error_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Mean residual: {np.mean(errors):.2f}, Std: {np.std(errors):.2f}")

Mean residual: 14.57, Std: 30.48


## Interpretation and Business Insight

- AQI is higher on weekdays due to traffic and industrial emissions
- Winter has worse air quality due to thermal inversions
- Pollution episodes (stagnant weather) create multi-day spikes
- Emission regulations create a slow improvement trend
- Weekend AQI drops reflect reduced traffic emissions

## Limitations

1. Synthetic — real AQI depends on meteorology, emissions inventory, terrain
2. No weather features (wind speed, temperature inversions)
3. Single pollutant aggregate — real AQI tracks PM2.5, O3, NO2 separately
4. No emission source data
5. No spatial variation (single monitoring station)

## How to Improve This Project

1. Add meteorological features (wind, temperature, humidity)
2. Model individual pollutants (PM2.5, O3, NO2)
3. Include emission source data (traffic counts, industrial activity)
4. Use spatial interpolation from multiple monitoring stations
5. Apply Chronos-Bolt for foundation-model forecasting

## Production Considerations

- Daily AQI forecasting for public health advisories
- Integration with weather forecast APIs
- Automated health alert triggering
- Traffic restriction decision support

## Common Mistakes

1. Ignoring meteorology — wind and inversions drive 80% of day-to-day variation
2. Using daily averages when hourly peaks matter for health
3. Not distinguishing pollutant sources
4. Treating AQI as stationary when there are trend changes

## Mini Challenge / Exercises

1. Add a synthetic wind speed feature and measure AQI correlation
2. Detect pollution episode events in the data
3. Build an AQI exceedance probability forecast (AQI > 100)
4. Compare winter vs summer AQI distributions

## Final Summary / Key Takeaways

1. Air quality forecasting is critical for public health
2. Weekly traffic and seasonal inversion patterns are predictable
3. Pollution episodes are the hardest to forecast (weather-dependent)
4. Real deployment needs meteorological inputs and multi-pollutant models
5. Forecast-based early warnings save lives

In [18]:
import json
metrics = {
    'project': 'Air Pollution Forecasting',
    'best_model': results_df.iloc[0]['model'],
    'best_MAE': float(results_df.iloc[0]['MAE']),
    'best_RMSE': float(results_df.iloc[0]['RMSE']),
    'best_MAPE': float(results_df.iloc[0]['MAPE']),
    'all_results': results_df.to_dict('records')
}
with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved.")
print("\n=== Air Pollution Forecasting — Complete ===")

Metrics saved.

=== Air Pollution Forecasting — Complete ===
